In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')
import joblib

In [22]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"✓ Uploaded: {filename}")
df = pd.read_csv('eff.csv')







Saving eff.csv to eff (3).csv
✓ Uploaded: eff (3).csv


In [23]:
if 'Employee ID' in df.columns:
    df = df.drop('Employee ID', axis=1)

X = df.drop('Attrition_Stayed', axis=1)
y = df['Attrition_Stayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [24]:

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)



In [25]:
# -------------------------
# Feature selection
# -------------------------
selector = SelectKBest(score_func=f_classif, k=20)
selector.fit(X_train_res, y_train_res)
selected_features = X_train.columns[selector.get_support()]
print(f"Selected {len(selected_features)} features: {list(selected_features)}")



Selected 20 features: ['Age', 'Years at Company', 'Number of Promotions', 'Distance from Home', 'Number of Dependents', 'Gender_Male', 'Work-Life Balance_Fair', 'Work-Life Balance_Good', 'Work-Life Balance_Poor', 'Overtime_Yes', 'Education Level_PhD', 'Marital Status_Married', 'Marital Status_Single', 'Job Level_Mid', 'Job Level_Senior', 'Company Size_Medium', 'Remote Work_Yes', 'Innovation Opportunities_Yes', 'Company Reputation_Good', 'Company Reputation_Poor']


In [26]:
# -------------------------
# Model definitions
# -------------------------
models_params = {
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=2000),
        'params': {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']}
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [100, 200], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5]}
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
        'params': {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}
    }
}



In [27]:
# -------------------------
# Train models with GridSearch
# -------------------------
best_models = {}
for name, mp in models_params.items():
    print(f"\nTraining {name}...")
    grid = GridSearchCV(mp['model'], mp['params'], cv=3, scoring='f1', n_jobs=-1, verbose=0)
    grid.fit(X_train_res[selected_features], y_train_res)
    best_models[name] = grid.best_estimator_
    print(f"  Best CV F1: {grid.best_score_:.4f}")




Training LogisticRegression...
  Best CV F1: 0.7469

Training RandomForest...
  Best CV F1: 0.7498

Training GradientBoosting...
  Best CV F1: 0.7555

Training XGBoost...
  Best CV F1: 0.7545


In [28]:


# -------------------------
# Evaluate models
# -------------------------
results = []
for name, model in best_models.items():
    y_pred = model.predict(X_test[selected_features])
    y_proba = model.predict_proba(X_test[selected_features])[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print("\nModel Results:")
print(results_df.to_string(index=False))

# -------------------------



Model Results:
             Model  Accuracy  Precision   Recall       F1  ROC_AUC
  GradientBoosting  0.754716   0.774369 0.748550 0.761241 0.846097
           XGBoost  0.753958   0.773582 0.747891 0.760520 0.845936
LogisticRegression  0.749346   0.768540 0.744333 0.756243 0.837367
      RandomForest  0.747969   0.771614 0.735108 0.752919 0.836127


In [30]:
# Stacking ensemble
# -------------------------
top3_models = results_df['Model'].iloc[:3].tolist()
estimators = [(name, best_models[name]) for name in top3_models if name != 'LogisticRegression']  # optional
stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=3,
    n_jobs=-1,
    passthrough=True
)
stack_model.fit(X_train_res[selected_features], y_train_res)

y_pred_stack = stack_model.predict(X_test[selected_features])
y_proba_stack = stack_model.predict_proba(X_test[selected_features])[:, 1]

stack_metrics = {
    'Model': 'StackingEnsemble',
    'Accuracy': accuracy_score(y_test, y_pred_stack),
    'Precision': precision_score(y_test, y_pred_stack),
    'Recall': recall_score(y_test, y_pred_stack),
    'F1': f1_score(y_test, y_pred_stack),
    'ROC_AUC': roc_auc_score(y_test, y_proba_stack)
}


results_df = pd.concat([results_df, pd.DataFrame([stack_metrics])], ignore_index=True).sort_values('F1', ascending=False)
best_model_name = results_df.iloc[0]['Model']
best_model = stack_model if best_model_name == 'StackingEnsemble' else best_models[best_model_name]
print(f"\nBest Model Selected: {best_model_name}")
print(f"Test F1 Score: {results_df.iloc[0]['F1']:.4f}")





Best Model Selected: GradientBoosting
Test F1 Score: 0.7612

Model saved as 'final_attrition_model.joblib'


In [31]:
# -------------------------
# Save model
# -------------------------
joblib.dump({
    'model': best_model,
    'selected_features': selected_features.tolist(),
    'performance': results_df.iloc[0].to_dict()
}, 'final_attrition_model.joblib')
print("\nModel saved as 'final_attrition_model.joblib'")



Model saved as 'final_attrition_model.joblib'
